# Note

This part has been migrated into Python scripts.

In [1]:
import json

# Replace with your file path
file_path = "employee_retention_articles.json"

# Read the JSON file
with open(file_path, "r", encoding="utf-8") as f:
    articles = json.load(f)

# Print the loaded data
print(articles)

[{'title': 'https://www.forbes.com/advisor/business/employee-retention-strategies/', 'url': 'https://www.forbes.com/advisor/business/employee-retention-strategies/', 'content': 'Subscribe To Newsletters Subscribe To Newsletters Trump Tracker Billionaires Billionaires View All Billionaires World\'s Billionaires Forbes 400 America\'s Richest Self-Made Women China\'s Richest India\'s Richest Indonesia\'s Richest Korea\'s Richest Thailand\'s Richest Japan\'s Richest Australia\'s Richest Taiwan\'s Richest Singapore\'s Richest Philippines\' Richest Hong Kong\'s Richest Malaysia\'s Richest Money Politics 2024 Election Innovation Innovation View All Innovation AI Big Data Cloud Cloud 100 Consumer Tech Creator Economy Cybersecurity Digital Transformation Enterprise Commerce Enterprise Security and Protection Enterprise Tech Enterprise Growth Strategy Future Of Work Gaming greenhouse Insights: How AI Can Help You Make Your Smartest Hire Yet Paid Program Healthcare Innovation Rules Retail Industr

In [ ]:
docs = []
docs_url = []
for a in articles:
    content = a.get("content")
    url = a.get('url')
    # If content is a dict or list, convert to string
    if isinstance(content, (dict, list)):
        content = str(content)

    # Skip empty content
    if not content:
        continue

    docs.append(content)
    docs_url.append(url)

## Simple implementation

In [10]:
from langchain_ollama.llms import OllamaLLM

from langchain_core.prompts import ChatPromptTemplate


model = OllamaLLM(model="llama3.2")

template = """ 
You are an expert in answerring questions about employee retention strategies based on the provided context.

Use the context to answer the question at the end. If the context does not provide enough information, say "I don't know".

Here are some relevant documents: {documents}

Here is the question to answer: {question}

"""

prompt = ChatPromptTemplate.from_template(template)


chain = prompt|model

results = chain.invoke({"documents":[docs], "question":"What are some effective employee retention strategies?"})

print(results)

Some effective employee retention strategies include:

1. **Regular pulse surveys**: Regularly solicit feedback from employees to understand their concerns and needs.
2. **Action planning**: Develop and implement plans to address issues and improve employee engagement.
3. **Retention Radar**: Use a predictive analytics tool to identify at-risk employees before they leave.
4. **Targeted retention strategies**: Equip managers with the data and tools they need to retain top talent.
5. **Career advancement opportunities**: Provide opportunities for growth and development to keep employees engaged.
6. **Recognition and feedback**: Regularly recognize and provide feedback to employees to show appreciation for their work.
7. **Meaningful company culture**: Foster a positive company culture that aligns with employee values and needs.

These strategies can help organizations prevent regrettable turnover, identify early warning signs of disengagement, and create a workplace where employees thriv

## Vector Database 

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
import os 
from langchain_ollama import OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

In [15]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

In [16]:
db_location = "./chrome_langchain_db"
add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    urls = []

    for i in range(len(docs)):
        doc = Document(page_content=docs[i], metadata={"source": docs_url[i]}, id = docs_url[i])
        documents.append(doc)
        urls.append(docs_url[i])

vector_store = Chroma(
    collection_name="employee_retention_articles",
    persist_directory=db_location,
    embedding_function=embeddings) 

if add_documents:
    vector_store.add_documents(documents = documents, ids = urls)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})




model = OllamaLLM(model="llama3.2")

template = """ 
You are an expert in answerring questions about employee retention strategies based on the provided context.

Use the context to answer the question at the end. If the context does not provide enough information, say "I don't know".

Here are some relevant documents: {documents}

Here is the question to answer: {question}

"""

prompt = ChatPromptTemplate.from_template(template)

chain = prompt|model

## Question 
question = "What are some effective employee retention strategies?"

retrieved_docs = retriever.invoke(question)

results = chain.invoke({"documents":[retrieved_docs], "question":question})

print(results)

According to the provided text, some effective employee retention strategies include:

1. Emphasizing work meaning and purpose.
2. Allowing employees to work from anywhere (telecommuting or remote work).
3. Offering opportunities for growth and advancement within the company.
4. Fostering a positive work environment and culture.
5. Providing support for managers who are sent abroad on assignment.
6. Addressing issues related to diversity, inclusion, and generational differences in the workplace.

These strategies aim to improve employee engagement, increase job satisfaction, and provide employees with a sense of fulfillment and purpose, ultimately leading to higher retention rates.
